# YSSY Wind Forecast Pipeline

End-to-end workflow for Sydney Airport (YSSY) wind forecasting:

```
Raw station .txt files (10 stations, half-hourly)
        |
        v
Part 1: Data Cleaning & Parquet
    1. Load 10 station files
    2. Linear interpolation (single-step NaN gaps)
    3. Wind speed/direction -> U/V components
    4. Spline interpolation (gaps up to 5 steps)
    5. Merge stations + save as Parquet
        |
        v
Part 2: Model Training
    1. Load Parquet
    2. Feature engineering (lag features + target construction)
    3. Train/Val/Test split (by time)
    4. Train MultiOutput LightGBM model (96 targets)
    5. Evaluate (MAE/MSE)
        |
        v
Part 3: Visual Display
    1. Forecast vs actual wind plots
    2. Performance vs lead time plot
```

In [ ]:
import pandas as pd
import numpy as np
import os, glob, time, warnings
from pathlib import Path
from datetime import timedelta
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from scipy.interpolate import UnivariateSpline
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

warnings.filterwarnings('ignore')
print("All imports successful.")

In [ ]:
# ============================================================
# Configuration
# ============================================================

# --- Paths ---
SCRIPT_DIR = Path.cwd()  # notebook's running directory
RAW_DATA_DIR = SCRIPT_DIR / "../scripts/YSSY-code/data/origional"
PARQUET_DIR = SCRIPT_DIR / "data"
PARQUET_PATH = PARQUET_DIR / "yssy_cleaned.parquet"

# --- Station list ---
STATION_IDS = ['BELL', 'MTB', 'YBTI', 'YCNK', 'YSBK', 'YSCN', 'YSNW', 'YSRI', 'YSSY', 'YSWG']
TARGET_STATION = 'YSSY'

# --- Interpolation ---
LINEAR_INTERP_COLS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']
SPLINE_INTERP_COLS = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
MAX_GAP = 5        # maximum NaN gap length for spline interpolation
CONTEXT = 6         # points to use on each side of a gap
SPLINE_K = 3        # cubic spline
SPLINE_S = 1        # smoothing factor

# --- Model ---
N_LAGS = 12          # past 6 hours of observations (12 x 30min)
N_STEPS = 48         # forecast 24 hours ahead (48 x 30min)
TRAIN_END = '2021-12-31'
VAL_START = '2022-01-01'
VAL_END = '2022-12-31'
TEST_START = '2023-01-01'
SUBSAMPLE_FRAC = 0.2 # fraction of training data to use (adjust for speed)

LGBM_PARAMS = {
    'n_estimators': 400,
    'learning_rate': 0.05,
    'num_leaves': 100,
    'max_depth': 10,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}
EARLY_STOPPING_ROUNDS = 30

# --- Plotting ---
N_SAMPLE_PLOTS = 4

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
print(f"Config loaded. Raw data: {RAW_DATA_DIR.resolve()}")
print(f"Parquet output: {PARQUET_PATH.resolve()}")

---
# Part 1: Data Cleaning & Parquet

Processes raw station `.txt` files through a three-stage interpolation pipeline, then merges all stations on common timestamps and saves as a single `.parquet` file.

In [ ]:
# --- Helper functions ---

def linear_interp_single_step(series):
    """Fill single-step NaN gaps with (prev + next) / 2."""
    result = series.copy()
    is_na = series.isna()
    n = len(series)
    for i in range(1, n - 1):
        if not is_na.iloc[i - 1] and is_na.iloc[i] and not is_na.iloc[i + 1]:
            result.iloc[i] = (series.iloc[i - 1] + series.iloc[i + 1]) / 2.0
    return result


def convert_wind_to_uv(df):
    """Convert wind_speed + wind_dir to u_component + v_component.
    Meteorological convention: u = -speed * sin(dir), v = speed * cos(dir).
    Drops the original wind_speed and wind_dir columns."""
    if 'wind_speed' not in df.columns or 'wind_dir' not in df.columns:
        return df
    ws = pd.to_numeric(df['wind_speed'], errors='coerce')
    wd = pd.to_numeric(df['wind_dir'], errors='coerce')
    dir_rad = np.deg2rad(wd.astype(float))
    df['u_component'] = -ws * np.sin(dir_rad)
    df['v_component'] = ws * np.cos(dir_rad)
    nan_mask = ws.isna() | wd.isna()
    df.loc[nan_mask, ['u_component', 'v_component']] = np.nan
    df.drop(columns=['wind_speed', 'wind_dir'], inplace=True)
    return df


def spline_interp_gap(series, gap_start, gap_end, ctx, k, s):
    """Interpolate a single NaN gap using cubic spline on surrounding context."""
    n = len(series)
    x_known, y_known = [], []
    for (lo, hi) in [(max(0, gap_start - ctx), gap_start - 1),
                      (gap_end + 1, min(n - 1, gap_end + ctx))]:
        if hi >= lo:
            chunk = series.iloc[lo:hi + 1].dropna()
            if not chunk.empty:
                x_known.extend([series.index.get_loc(i) for i in chunk.index])
                y_known.extend(chunk.tolist())
    if len(x_known) < k + 1:
        return None
    x_arr, y_arr = np.array(x_known), np.array(y_known)
    order = np.argsort(x_arr)
    ux, ui = np.unique(x_arr[order], return_index=True)
    if len(ux) < k + 1:
        return None
    try:
        sp = UnivariateSpline(ux, y_arr[order][ui], k=k, s=s)
        ilocs = np.arange(gap_start, gap_end + 1)
        return pd.Series(sp(ilocs), index=series.index[ilocs])
    except Exception:
        return None


def spline_interp_series(series):
    """Apply cubic spline interpolation to all NaN gaps up to MAX_GAP length."""
    result = series.copy()
    is_na = series.isna()
    gap_start = -1
    for i in range(len(series)):
        if is_na.iloc[i] and gap_start == -1:
            gap_start = i
        elif (not is_na.iloc[i] or i == len(series) - 1) and gap_start != -1:
            gap_end = (i - 1) if not is_na.iloc[i] else i
            gap_len = gap_end - gap_start + 1
            if 0 < gap_len <= MAX_GAP:
                vals = spline_interp_gap(series, gap_start, gap_end, CONTEXT, SPLINE_K, SPLINE_S)
                if vals is not None and not vals.empty:
                    result.loc[vals.index] = vals.values
            gap_start = -1
    return result


print("All helper functions defined.")

In [ ]:
# --- Process all stations ---

cleaned = {}
total_start = time.time()

for sid in STATION_IDS:
    filepath = RAW_DATA_DIR / f"{sid}.txt"
    if not filepath.exists():
        print(f"  {sid}: file not found, skipping.")
        continue

    t0 = time.time()
    print(f"\n--- {sid} ---")
    df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'])
    print(f"  Loaded {len(df):,} rows")

    # 1. Linear interpolation
    for col in LINEAR_INTERP_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = linear_interp_single_step(df[col])
    print(f"  Linear interp done")

    # 2. Wind -> U/V conversion
    df = convert_wind_to_uv(df)
    print(f"  UV conversion done")

    # 3. Spline interpolation
    df = df.set_index('timestamp')
    for col in SPLINE_INTERP_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = spline_interp_series(df[col])
    df = df.reset_index()
    print(f"  Spline interp done")

    cleaned[sid] = df
    print(f"  Done in {time.time() - t0:.1f}s. Columns: {list(df.columns)}")

print(f"\n=== All stations processed in {time.time() - total_start:.1f}s ===")
print(f"Stations processed: {list(cleaned.keys())}")

In [ ]:
# --- Merge all stations on timestamp ---

t0 = time.time()
dfs = []
for sid, df in cleaned.items():
    df = df.set_index('timestamp')
    # Prefix all columns with station ID
    df.columns = [f"{sid}_{c}" for c in df.columns]
    dfs.append(df)

merged = pd.concat(dfs, axis=1)
merged = merged.sort_index()
merged = merged.reset_index()  # timestamp becomes a column

n_cols = len(merged.columns)
n_rows = len(merged)
print(f"Merged: {n_rows:,} rows x {n_cols} columns")
print(f"Time range: {merged['timestamp'].min()} to {merged['timestamp'].max()}")
print(f"Merge took {time.time() - t0:.1f}s")

In [ ]:
# --- Save as Parquet ---

t0 = time.time()
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
merged.to_parquet(PARQUET_PATH, index=False)
size_mb = os.path.getsize(PARQUET_PATH) / 1e6
print(f"Saved: {PARQUET_PATH}")
print(f"Size: {size_mb:.1f} MB")
print(f"Done in {time.time() - t0:.1f}s")

---
# Part 2: Model Training

Loads the cleaned Parquet file, constructs lag features and forecast targets, then trains a **MultiOutput LightGBM model** to predict YSSY wind U/V components 0.5 to 24 hours ahead (48 half-hourly steps x 2 components = **96 targets**).

In [ ]:
# --- Load Parquet ---

t0 = time.time()
df = pd.read_parquet(PARQUET_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Time range: {df.index.min()} to {df.index.max()}")
print(f"Done in {time.time() - t0:.1f}s")

In [ ]:
# --- Feature Engineering ---

t0 = time.time()

# 1. YSSY U/V lag features (past 6 hours)
for lag in range(N_LAGS):
    df[f'yssy_u_lag{lag}'] = df[f'{TARGET_STATION}_u_component'].shift(lag)
    df[f'yssy_v_lag{lag}'] = df[f'{TARGET_STATION}_v_component'].shift(lag)
print(f"Lag features created: {N_LAGS * 2} columns")

# 2. Target columns (future 24 hours of YSSY U/V)
target_cols = []
for step in range(1, N_STEPS + 1):
    u_col = f'target_u_t{step}'
    v_col = f'target_v_t{step}'
    df[u_col] = df[f'{TARGET_STATION}_u_component'].shift(-step)
    df[v_col] = df[f'{TARGET_STATION}_v_component'].shift(-step)
    target_cols.extend([u_col, v_col])
print(f"Target columns created: {len(target_cols)}")

# 3. Drop rows where any target is NaN (end of dataset, insufficient future data)
before_drop = len(df)
df = df.dropna(subset=target_cols)
print(f"Rows after dropping incomplete targets: {len(df):,} (dropped {before_drop - len(df):,})")

# 4. Build feature set: YSSY lag features + other stations' current observations
feature_cols = [f'yssy_u_lag{lag}' for lag in range(N_LAGS)] + \
              [f'yssy_v_lag{lag}' for lag in range(N_LAGS)]

other_vars = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
for sid in STATION_IDS:
    if sid == TARGET_STATION:
        continue
    for var in other_vars:
        col = f'{sid}_{var}'
        if col in df.columns:
            feature_cols.append(col)

print(f"Total features: {len(feature_cols)}")
print(f"Done in {time.time() - t0:.1f}s")

In [ ]:
# --- Train / Validation / Test split (by time) ---

mask_train = df.index <= TRAIN_END
mask_val = (df.index >= VAL_START) & (df.index <= VAL_END)
mask_test = df.index >= TEST_START

X_train, y_train = df.loc[mask_train, feature_cols], df.loc[mask_train, target_cols]
X_val, y_val = df.loc[mask_val, feature_cols], df.loc[mask_val, target_cols]
X_test, y_test = df.loc[mask_test, feature_cols], df.loc[mask_test, target_cols]

# Optional subsample for faster training
if SUBSAMPLE_FRAC < 1.0 and len(X_train) > 0:
    X_train = X_train.sample(frac=SUBSAMPLE_FRAC, random_state=42)
    y_train = y_train.loc[X_train.index]

print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {X_train.shape[1]} | Targets: {y_train.shape[1]}")

In [ ]:
# --- Train MultiOutput LightGBM Model ---

print(f"Training MultiOutputRegressor with {LGBM_PARAMS['n_estimators']} max estimators...")
base_model = lgb.LGBMRegressor(**LGBM_PARAMS)
model = MultiOutputRegressor(base_model, n_jobs=-1)

t0 = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"Training done in {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
# --- Evaluate on Test Set ---

t0 = time.time()
y_pred = model.predict(X_test)

overall_mae = mean_absolute_error(y_test, y_pred)
overall_mse = mean_squared_error(y_test, y_pred)
print(f"Overall Test MAE: {overall_mae:.4f}")
print(f"Overall Test MSE: {overall_mse:.4f}")

# Per-step metrics
rows = []
for step in range(1, N_STEPS + 1):
    u_col, v_col = f'target_u_t{step}', f'target_v_t{step}'
    idx_u, idx_v = target_cols.index(u_col), target_cols.index(v_col)
    mae_u = mean_absolute_error(y_test.iloc[:, idx_u], y_pred[:, idx_u])
    mae_v = mean_absolute_error(y_test.iloc[:, idx_v], y_pred[:, idx_v])
    mse_u = mean_squared_error(y_test.iloc[:, idx_u], y_pred[:, idx_u])
    mse_v = mean_squared_error(y_test.iloc[:, idx_v], y_pred[:, idx_v])
    rows.append({'step': step, 'hours': step * 0.5,
                 'MAE_u': mae_u, 'MAE_v': mae_v, 'Avg_MAE': (mae_u + mae_v) / 2,
                 'MSE_u': mse_u, 'MSE_v': mse_v, 'Avg_MSE': (mse_u + mse_v) / 2})

metrics_df = pd.DataFrame(rows)
print(f"Evaluation done in {time.time() - t0:.1f}s")
display(metrics_df.round(4).head(12))

---
# Part 3: Visual Display

In [ ]:
# --- Helper: U/V -> wind speed & direction ---

def uv_to_ws_wd(u, v):
    """Convert U/V components to wind speed (knots) and direction (degrees)."""
    u, v = np.asarray(u, float), np.asarray(v, float)
    ws = np.sqrt(u**2 + v**2)
    wd = (270 - np.degrees(np.arctan2(-v, u))) % 360
    wd[ws < 0.1] = 0
    return ws, wd


def plot_forecast_sample(idx, X_row, y_true_row, y_pred_row, ts):
    """Plot observed data plus forecasted wind for a single sample."""
    plot_start = ts - timedelta(hours=24)
    plot_end = ts + timedelta(hours=24)

    # Observed data from the cleaned DataFrame (still available from Part 2)
    obs_slice = df.loc[plot_start:plot_end]
    if obs_slice.empty:
        print(f"  No obs data for {ts}")
        return

    obs_t = obs_slice.index
    obs_temp = obs_slice.get(f'{TARGET_STATION}_air_temp', pd.Series(np.nan, index=obs_slice.index)).values
    obs_dewp = obs_slice.get(f'{TARGET_STATION}_dew_point', pd.Series(np.nan, index=obs_slice.index)).values
    obs_u = obs_slice.get(f'{TARGET_STATION}_u_component', pd.Series(np.nan, index=obs_slice.index)).values
    obs_v = obs_slice.get(f'{TARGET_STATION}_v_component', pd.Series(np.nan, index=obs_slice.index)).values
    obs_ws, obs_wd = uv_to_ws_wd(obs_u, obs_v)

    # Forecast future times
    fcst_t = [ts + timedelta(minutes=30 * i) for i in range(1, N_STEPS + 1)]
    fcst_u = y_pred_row[0::2]  # every other element is u
    fcst_v = y_pred_row[1::2]  # every other element is v
    fcst_ws, fcst_wd = uv_to_ws_wd(fcst_u, fcst_v)

    fig, ax1 = plt.subplots(figsize=(18, 9))
    fig.suptitle(f"YSSY Wind Forecast — Anchor: {ts.strftime('%Y-%m-%d %H:%M')}", fontsize=15)

    ax1.plot(obs_t, obs_temp, 'r-', alpha=0.7, label='Obs Temp (C)')
    ax1.plot(obs_t, obs_dewp, 'b-', alpha=0.7, label='Obs Dewpoint (C)')
    ax1.plot(obs_t, obs_ws, '-', color='gray', lw=2.5, alpha=0.7, label='Obs Wind Speed (kt)')
    ax1.plot(fcst_t, fcst_ws, '-', color='black', lw=3.5, label='Fcst Wind Speed')
    ax1.set_ylabel('Temp (C) / Wind Speed (kt)')
    ax1.grid(True, ls=':', alpha=0.5)
    ax1.yaxis.set_major_locator(mticker.MultipleLocator(5))

    # Dynamic Y limit
    all_vals = np.concatenate([obs_temp[~np.isnan(obs_temp)], obs_dewp[~np.isnan(obs_dewp)],
                                obs_ws[~np.isnan(obs_ws)], fcst_ws[~np.isnan(fcst_ws)]])
    y_lo = min(0, np.floor(np.nanmin(all_vals) / 5) * 5) if len(all_vals) > 0 else 0
    y_hi = max(30, np.ceil(np.nanmax(all_vals) / 5) * 5) if len(all_vals) > 0 else 40
    ax1.set_ylim(y_lo, y_hi)

    ax2 = ax1.twinx()
    ax2.scatter(obs_t, obs_wd, marker='o', color='darkgray', s=40, alpha=0.6, label='Obs Wind Dir')
    ax2.scatter(fcst_t, fcst_wd, marker='X', color='black', s=55, label='Fcst Wind Dir')
    ax2.set_ylabel('Wind Direction (deg)')
    ax2.set_ylim(0, 360)
    ax2.yaxis.set_major_locator(mticker.MultipleLocator(45))

    ax1.set_xlim(plot_start, plot_end)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d-%b'))
    ax1.xaxis.set_major_locator(mdates.HourLocator(interval=3))
    ax1.axvline(ts, color='k', ls=':', lw=1, alpha=0.5)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    plt.tight_layout(rect=[0, 0.06, 1, 0.94])
    plt.show()

print("Plotting function defined.")

In [ ]:
# --- Forecast vs Actual: Random Test Samples ---

np.random.seed(42)
n_plots = min(N_SAMPLE_PLOTS, len(X_test))
sample_idx = np.random.choice(len(X_test), n_plots, replace=False)

print(f"Plotting {n_plots} random test samples...\n")
for i, idx in enumerate(sample_idx):
    ts = X_test.index[idx]
    plot_forecast_sample(idx, X_test.iloc[idx], y_test.iloc[idx], y_pred[idx], ts)

In [ ]:
# --- Model Performance vs Forecast Lead Time ---

fig, (ax_mae, ax_mse) = plt.subplots(1, 2, figsize=(20, 7))

hours = metrics_df['hours']

ax_mae.plot(hours, metrics_df['MAE_u'], 'r--o', markersize=3, label='MAE U')
ax_mae.plot(hours, metrics_df['MAE_v'], 'b--s', markersize=3, label='MAE V')
ax_mae.plot(hours, metrics_df['Avg_MAE'], 'k-', linewidth=2.5, label='Avg MAE')
ax_mae.set_xlabel('Forecast Lead Time (hours)')
ax_mae.set_ylabel('MAE (knots)')
ax_mae.set_xticks(np.arange(0, 26, 2))
ax_mae.set_ylim(bottom=0)
ax_mae.grid(True, ls=':', alpha=0.5)
ax_mae.set_title('MAE vs Lead Time', fontsize=14)
ax_mae.legend()

ax_mse.plot(hours, metrics_df['MSE_u'], 'r--o', markersize=3, label='MSE U')
ax_mse.plot(hours, metrics_df['MSE_v'], 'b--s', markersize=3, label='MSE V')
ax_mse.plot(hours, metrics_df['Avg_MSE'], 'k-', linewidth=2.5, label='Avg MSE')
ax_mse.set_xlabel('Forecast Lead Time (hours)')
ax_mse.set_ylabel('MSE')
ax_mse.set_xticks(np.arange(0, 26, 2))
ax_mse.set_ylim(bottom=0)
ax_mse.grid(True, ls=':', alpha=0.5)
ax_mse.set_title('MSE vs Lead Time', fontsize=14)
ax_mse.legend()

fig.suptitle(f'LightGBM MultiOutput Model — {len(feature_cols)} features, {len(target_cols)} targets', fontsize=16)
plt.tight_layout()
plt.show()